# 01 · Data check (Week 1)

Goal: confirm the cleaned BTS data can answer the locked research question **before** any analysis.

> **Which airports or carriers show disproportionately high carrier-attributable delay, and does time-of-day or day-of-week make it worse?**

Prerequisite: `python scripts/download_bts.py && python scripts/build_dataset.py` (see README).

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from aviation_delays import data, metrics, charts

pd.set_option("display.float_format", "{:,.2f}".format)
DATA_PATH = Path(os.environ.get("AVIATION_DATA", data.PROCESSED_PATH))
flights = data.load_processed(DATA_PATH)
af = data.analysis_frame(flights)

## Cleaning summary
Rows are flagged, not dropped. Delay metrics use completed flights with a usable cause breakdown (`data.analysis_frame`).

In [2]:
summary = data.cleaning_summary(flights)
summary["analysis_rows"] = len(af)
summary["analysis_share_of_rows"] = round(len(af) / len(flights), 4)
summary.to_frame("value")

,value
rows,14080680
date_min,2024-01-01
date_max,2025-12-31
carriers,15
origin_airports,358
completed,13844732
cancelled,199191
diverted,36757
arr_delayed_15,2984604
delayed_cause_missing,2


## Answerability checks

| # | Check | Pass if |
|---|---|---|
| 1 | Cause fields are populated for delayed flights | < 2% of ≥15-min-late flights missing causes |
| 2 | Enough carriers to compare | ≥ 8 carriers, each ≥ 50k completed flights |
| 3 | Enough airports to compare | ≥ 30 origin airports with ≥ 20k completed flights |
| 4 | Time-of-day / day-of-week cells have volume | every hour 05–23 × weekday cell ≥ 1k flights |
| 5 | Carrier delay is a material share of cause-coded minutes | ≥ 15% (otherwise the question is not very interesting) |

In [3]:
delayed = int(flights["arr_del15"].sum())
missing_share = flights["cause_missing"].sum() / delayed
carrier_n = af.groupby("carrier", observed=True).size().sort_values(ascending=False)
airport_n = af.groupby("origin", observed=True).size().sort_values(ascending=False)
cell_n = af[af["dep_hour"].between(5, 23)].groupby(["dep_hour", "day_of_week"], observed=True).size()
carrier_share = metrics.network_summary(af)["carrier_share"]

checks = pd.DataFrame(
    [
        ("1 cause fields populated", f"{missing_share:.2%} of delayed flights missing causes", missing_share < 0.02),
        ("2 carriers", f"{(carrier_n >= 50_000).sum()} carriers with >= 50k flights", (carrier_n >= 50_000).sum() >= 8),
        ("3 airports", f"{(airport_n >= 20_000).sum()} airports with >= 20k flights", (airport_n >= 20_000).sum() >= 30),
        ("4 time cells", f"smallest hour x weekday cell: {cell_n.min():,} flights", cell_n.min() >= 1_000),
        ("5 carrier share", f"carrier = {carrier_share:.1%} of cause-coded minutes", carrier_share >= 0.15),
    ],
    columns=["check", "result", "pass"],
)
checks

,check,result,pass
0,1 cause fields populated,0.00% of delayed flights missing causes,True
1,2 carriers,15 carriers with >= 50k flights,True
2,3 airports,102 airports with >= 20k flights,True
3,4 time cells,"smallest hour x weekday cell: 14,363 flights",True
4,5 carrier share,carrier = 33.4% of cause-coded minutes,True


In [4]:
# Volume per carrier (for the min-volume threshold used in 02_analysis)
carrier_n.to_frame("completed_flights")

,completed_flights
carrier,
WN,2781434
DL,2010253
AA,1918957
OO,1559088
UA,1532195
YX,629350
MQ,567152
AS,482250
B6,462386


In [5]:
# Cause coverage by month: a sudden gap would mean a reporting problem in that month
flights.assign(month=flights["date"].dt.strftime("%Y-%m")).groupby("month").agg(
    flights=("status", "size"),
    cancelled_rate=("status", lambda s: (s == "cancelled").mean()),
    delayed_15=("arr_del15", "sum"),
    cause_missing=("cause_missing", "sum"),
)

,flights,cancelled_rate,delayed_15,cause_missing
month,,,,
2024-01,547271,0.04,126410,0
2024-02,519221,0.01,82036,0
2024-03,591767,0.01,122032,0
2024-04,582185,0.01,109948,0
2024-05,609743,0.01,158276,0
2024-06,611132,0.01,149945,0
2024-07,634613,0.03,181865,2
2024-08,619025,0.02,141812,0
2024-09,582622,0.01,89678,0


## Decision

**Answerable as locked: yes** (run on BTS Jan 2024 – Dec 2025, 14.08M flights).

- All five checks pass: 2 of 2.98M delayed flights lack a cause breakdown, 15 carriers each have ≥ 150k flights, 102 airports have ≥ 20k departures, the smallest hour × weekday cell has 14k flights, and carrier delay is 33% of cause-coded minutes.
- Watch-outs carried into 02: January 2024 (winter storms) and July 2024 (CrowdStrike IT outage, 19 Jul, which hit Delta hardest) have high cancellation rates. 02 re-checks the findings with the outage week excluded.
- Endeavor (9E) stops reporting separately in 2025.
- Thresholds carried into 02 unchanged: carriers ≥ 50k flights, airports ≥ 20k departures.